# Hourly Time Series Forecasting using Facebook's Prophet
![](https://miro.medium.com/max/964/0*tVCene42rgUTNv9Q.png)

In this notebook we will use facebook's prophet package to forecast hourly energy use.

# Background on the Types of Time Series Data
![img](https://miro.medium.com/max/1400/1*V_RKPeIxCB9CS_2SsLyKXw.jpeg)

In [ ]:
!pip install prophet

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from prophet import Prophet

from sklearn.metrics import mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

plt.style.use('ggplot')
plt.style.use('fivethirtyeight')

def mean_absolute_percentage_error(y_true, y_pred):
    """Calculates MAPE given y_true and y_pred"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Data Loading
We are now using the datasets available in the environment. The primary forecasting target will be loaded from `time_series_global.csv`, and we will supplement it with category-specific data from the other three files.

In [ ]:
import pandas as pd

# Loading the primary time series dataset from the available files
df = pd.read_csv('/content/time_series_global.csv',
                 index_col=[0],
                 parse_dates=[0])

# Rename columns to Prophet standards if necessary (ds, y)
if 'ds' not in df.columns and df.index.name != 'ds':
    df.index.name = 'ds'

display(df.head())
print(f"\nAvailable columns: {df.columns.tolist()}")

In [ ]:
color_pal = sns.color_palette()
df.plot(style='.',
          figsize=(10, 5),
          ms=1,
          color=color_pal[0],
          title='Sales Data')
plt.show()

# Time Series Features

In [ ]:
from pandas.api.types import CategoricalDtype

cat_type = CategoricalDtype(categories=['Monday','Tuesday',
                                        'Wednesday',
                                        'Thursday','Friday',
                                        'Saturday','Sunday'],
                            ordered=True)

def create_features(df, label=None):
    """
    Creates time series features from datetime index.
    """
    df = df.copy()
    df['date'] = df.index
    df['hour'] = df['date'].dt.hour
    df['dayofweek'] = df['date'].dt.dayofweek
    df['weekday'] = df['date'].dt.day_name()
    df['weekday'] = df['weekday'].astype(cat_type)
    df['quarter'] = df['date'].dt.quarter
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['dayofyear'] = df['date'].dt.dayofyear
    df['dayofmonth'] = df['date'].dt.day
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
    df['date_offset'] = (df.date.dt.month*100 + df.date.dt.day - 320)%1300

    df['season'] = pd.cut(df['date_offset'], [0, 300, 602, 900, 1300],
                          labels=['Spring', 'Summer', 'Fall', 'Winter']
                   )
    X = df[['hour','dayofweek','quarter','month','year',
           'dayofyear','dayofmonth','weekofyear','weekday',
           'season']]
    if label:
        y = df[label]
        return X, y
    return X

X, y = create_features(df, label='y')
features_and_target = pd.concat([X, y], axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=features_and_target.dropna(),
            x='weekday',
            y='y',
            hue='season',
            ax=ax,
            linewidth=1)
ax.set_title('Sales by Day of Week')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Sales (y)')
ax.legend(bbox_to_anchor=(1, 1))
plt.show()

# Train / Test Split

In [ ]:
split_date = '2018-01-01'
df_train = df.loc[df.index <= split_date].copy()
df_test = df.loc[df.index > split_date].copy()

# Plot train and test - selecting only 'y' column to avoid overlap errors
df_test[['y']] \
    .rename(columns={'y': 'TEST SET'}) \
    .join(df_train[['y']].rename(columns={'y': 'TRAINING SET'}),
          how='outer') \
    .plot(figsize=(10, 5), title='Train/Test Split', style='.', ms=1)
plt.show()

# Simple Prophet Model
- Prophet model expects the dataset to be named a specific way. We will rename our dataframe columns before feeding it into the model.
    - Datetime column named: `ds`
    - target : `y`

In [ ]:
# Format data for prophet model using ds and y
df_train_prophet = df_train.reset_index() \
    .rename(columns={'ds':'ds',
                     'y':'y'})
# Note: If your index was already 'ds', reset_index() handles it.

In [ ]:
%%time
model = Prophet()
model.fit(df_train_prophet)

In [ ]:
# Predict on test set with model
df_test_prophet = df_test.reset_index() \
    .rename(columns={'ds':'ds',
                     'y':'y'})

df_test_fcst = model.predict(df_test_prophet)

In [ ]:
df_test_fcst.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fig = model.plot(df_test_fcst, ax=ax)
ax.set_title('Prophet Forecast')
plt.show()

In [ ]:
fig = model.plot_components(df_test_fcst)
plt.show()

# Compare Forecast to Actuals

In [ ]:
# Plot the forecast with the actuals
f, ax = plt.subplots(figsize=(15, 5))
ax.scatter(df_test.index, df_test['y'], color='r')
fig = model.plot(df_test_fcst, ax=ax)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(df_test.index, df_test['y'], color='r')
fig = model.plot(df_test_fcst, ax=ax)
ax.set_xbound(lower=pd.to_datetime('2018-01-01'),
              upper=pd.to_datetime('2018-02-01'))
ax.set_title('January 2018 Forecast vs Actuals')
plt.show()

In [ ]:
# Plot the forecast with the actuals for the first week
f, ax = plt.subplots(figsize=(15, 5))
ax.scatter(df_test.index, df_test['y'], color='r')
fig = model.plot(df_test_fcst, ax=ax)
ax.set_xbound(lower=pd.to_datetime('2018-01-01'), upper=pd.to_datetime('2018-01-08'))
ax.set_title('First Week of January 2018 Forecast vs Actuals')
plt.show()

# Evaluate the model with Error Metrics

In [ ]:
np.sqrt(mean_squared_error(y_true=df_test['y'],
                   y_pred=df_test_fcst['yhat']))

In [ ]:
mean_absolute_error(y_true=df_test['y'],
                   y_pred=df_test_fcst['yhat'])

In [ ]:
mean_absolute_percentage_error(y_true=df_test['y'],
                   y_pred=df_test_fcst['yhat'])

# Adding Holidays

Next we will see if adding holiday indicators will help the accuracy of the model. Prophet comes with a Holiday Effects parameter that can be provided to the model prior to training.

We will use the built in pandas USFederalHolidayCalendar to pull the list of holidays

In [ ]:
# Menggunakan Fitur Hari Libur dari Dataset
# Karena dataset time_series_global.csv sudah menyertakan indikator is_holiday,
# kita akan langsung menggunakannya sebagai regressor tambahan dalam model Prophet.
from prophet import Prophet

# Menyiapkan data training
df_train_prophet = df_train.reset_index().rename(columns={'ds':'ds', 'y':'y'})

# Inisialisasi model
model_with_holidays = Prophet()

# Menambahkan is_holiday dari dataset sebagai regressor
model_with_holidays.add_regressor('is_holiday')

# Fit model
model_with_holidays.fit(df_train_prophet)

In [ ]:
# Menyiapkan data test
df_test_prophet = df_test.reset_index().rename(columns={'ds':'ds', 'y':'y'})

# Prediksi menggunakan kolom is_holiday yang ada di test set
df_test_fcst_with_hols = model_with_holidays.predict(df_test_prophet)

In [ ]:
fig = model_with_holidays.plot_components(
    df_test_fcst_with_hols)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(df_test.index, df_test['y'], color='r')
fig = model_with_holidays.plot(df_test_fcst_with_hols, ax=ax)
ax.set_xbound(lower=pd.to_datetime('2018-07-01'),
              upper=pd.to_datetime('2018-07-07'))
ax.set_title('July Forecast with Holidays vs Actual')
plt.show()

In [ ]:
np.sqrt(mean_squared_error(y_true=df_test['y'],
                   y_pred=df_test_fcst_with_hols['yhat']))

In [ ]:
mean_absolute_error(y_true=df_test['y'],
                   y_pred=df_test_fcst_with_hols['yhat'])

In [ ]:
mean_absolute_percentage_error(y_true=df_test['y'],
                   y_pred=df_test_fcst_with_hols['yhat'])

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar as calendar

# 1. Pastikan data tersedia menggunakan dataset yang benar
try:
    df.head()
except NameError:
    # Menggunakan time_series_global.csv sebagai sumber data utama
    df = pd.read_csv('/content/time_series_global.csv', index_col=[0], parse_dates=[0])

split_date = '2018-01-01'
df_train = df.loc[df.index <= split_date].copy()
df_test = df.loc[df.index > split_date].copy()
df_train_prophet = df_train.reset_index().rename(columns={'ds':'ds', 'y':'y'})
df_test_prophet = df_test.reset_index().rename(columns={'ds':'ds', 'y':'y'})

# 2. Inisialisasi Ulang Model dan Prediksi jika variabel hilang
try:
    model_with_holidays
except NameError:
    cal = calendar()
    hols = cal.holidays(start=df.index.min(), end=df.index.max(), return_name=True)
    holiday_df = pd.DataFrame(data=hols, columns=['holiday']).reset_index().rename(columns={'index':'ds'})
    model_with_holidays = Prophet(holidays=holiday_df)
    model_with_holidays.fit(df_train_prophet)

df_test_fcst_with_hols = model_with_holidays.predict(df=df_test_prophet)

# 3. Analisis perbandingan
comparison = pd.DataFrame({
    'Actual': df_test['y'].values,
    'Predicted': df_test_fcst_with_hols['yhat'].values
}, index=df_test.index)

print("Statistik Deskriptif:")
display(comparison.describe())

# Analisis Error
comparison['Error'] = comparison['Actual'] - comparison['Predicted']
comparison['Abs_Percentage_Error'] = (np.abs(comparison['Error']) / (comparison['Actual'] + 1e-6)) * 100

print("\nAnalisis Nilai Ekstrim:")
print(f"Jumlah nilai 0 pada data aktual: {(df_test['y'] == 0).sum()}")
print(f"Nilai minimum pada data aktual: {df_test['y'].min()}")
print(f"Median dari Absolute Percentage Error: {comparison['Abs_Percentage_Error'].median():.2f}%")

print("\nKesimpulan:")
print("Model telah diperbaiki untuk menggunakan dataset global yang tersedia.")

# Predict into the Future

We can use the built in `make_future_dataframe` method to build our future dataframe and make predictions.

In [ ]:
future = model.make_future_dataframe(periods=365*24, freq='h', include_history=False)
forecast = model_with_holidays.predict(future)

In [ ]:
forecast[['ds','yhat']].head()

# Lanjutan

Jika kolom kategori (seperti `category` atau `item_type`) tersedia, kita bisa melakukan iterasi untuk melatih model Prophet pada setiap kategori tersebut. Namun, dalam kode yang dijalankan sejauh ini, model hanya dilatih untuk satu target agregat (`y`).

### Exploring Additional Datasets for Model Improvement
We will inspect the available files to identify useful features for `add_regressor` or for segmenting our models.

In [ ]:
import pandas as pd

# Load samples of the datasets to understand their structure
files = [
    '/content/outlier_summary.csv',
    '/content/inventory_summary.csv',
    '/content/time_series_global.csv',
    '/content/time_series_category_feature_engineered.csv'
]

for file in files:
    print(f"--- {file} ---")
    try:
        tmp_df = pd.read_csv(file)
        display(tmp_df.head())
        print(tmp_df.columns.tolist())
    except Exception as e:
        print(f"Could not read {file}: {e}")
    print("\n")

### Perbaikan Model: Multi-Category Forecasting dengan Feature Engineering
Kita akan menggunakan dataset `time_series_category_feature_engineered.csv` untuk melatih model Prophet yang spesifik untuk setiap kategori produk.

In [ ]:
import pandas as pd
from prophet import Prophet

# Load dataset yang sudah memiliki fitur
df_cat = pd.read_csv('/content/time_series_category_feature_engineered.csv', parse_dates=['ds'])

categories = df_cat['Category'].unique()
results = {}

for cat in categories:
    print(f"Training model for category: {cat}...")

    # Filter data per kategori
    df_sub = df_cat[df_cat['Category'] == cat].copy()

    # Inisialisasi model dengan holiday dan regressor tambahan jika ada
    m = Prophet(holidays=holiday_df if 'holiday_df' in locals() else None)

    # Menambahkan is_holiday sebagai regressor jika bukan bagian dari holiday_df
    if 'is_holiday' in df_sub.columns:
        m.add_regressor('is_holiday')

    # Split train/test (80/20)
    train_size = int(len(df_sub) * 0.8)
    train_df = df_sub.iloc[:train_size]
    test_df = df_sub.iloc[train_size:]

    m.fit(train_df[['ds', 'y', 'is_holiday']])

    # Forecast
    forecast = m.predict(test_df[['ds', 'is_holiday']])

    # Simpan hasil untuk evaluasi
    results[cat] = {
        'model': m,
        'forecast': forecast,
        'actual': test_df['y'].values
    }

print("\nSelesai! Model untuk setiap kategori telah dilatih.")

### Evaluasi Akurasi Per Kategori
Kita akan menghitung error untuk masing-masing kategori untuk melihat model mana yang paling akurat.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
import numpy as np

performance = []

for cat, data in results.items():
    actual = data['actual']
    predicted = data['forecast']['yhat'].values

    mae = mean_absolute_error(actual, predicted)

    # Menggunakan Median Absolute Error karena banyak nilai 0 (sparse data)
    med_ae = np.median(np.abs(actual - predicted))

    performance.append({
        'Category': cat,
        'MAE': mae,
        'Median_AE': med_ae
    })

performance_df = pd.DataFrame(performance)
print("Evaluasi Akurasi Per Kategori:")
display(performance_df)

# Visualisasi perbandingan kategori
for cat in categories:
    fig = results[cat]['model'].plot(results[cat]['forecast'])
    plt.title(f'Forecast vs Actual for {cat}')
    plt.show()

### Menambahkan Lagged Features dan Rolling Mean sebagai Regressor
Fitur `lag_1` dan `rolling_mean_7` akan ditambahkan menggunakan metode `add_regressor` untuk meningkatkan kemampuan model dalam menangkap tren jangka pendek.

In [ ]:
results_enhanced = {}

for cat in categories:
    print(f"Training enhanced model for: {cat}...")
    df_sub = df_cat[df_cat['Category'] == cat].copy()

    # Menghapus baris dengan NaN yang biasanya muncul karena pembuatan fitur lag/rolling
    df_sub = df_sub.dropna(subset=['lag_1', 'rolling_mean_7'])

    m = Prophet(holidays=holiday_df if 'holiday_df' in locals() else None)

    # Tambahkan regressor
    if 'is_holiday' in df_sub.columns:
        m.add_regressor('is_holiday')
    m.add_regressor('lag_1')
    m.add_regressor('rolling_mean_7')

    # Split data
    train_size = int(len(df_sub) * 0.8)
    train_df = df_sub.iloc[:train_size]
    test_df = df_sub.iloc[train_size:]

    # Fit model dengan regressor tambahan
    m.fit(train_df[['ds', 'y', 'is_holiday', 'lag_1', 'rolling_mean_7']])

    # Forecast
    forecast = m.predict(test_df[['ds', 'is_holiday', 'lag_1', 'rolling_mean_7']])

    results_enhanced[cat] = {
        'model': m,
        'forecast': forecast,
        'actual': test_df['y'].values
    }

# Evaluasi Akurasi Baru
perf_enhanced = []
for cat, data in results_enhanced.items():
    mae = mean_absolute_error(data['actual'], data['forecast']['yhat'].values)
    med_ae = np.median(np.abs(data['actual'] - data['forecast']['yhat'].values))
    perf_enhanced.append({'Category': cat, 'MAE_Enhanced': mae, 'MedAE_Enhanced': med_ae})

display(pd.DataFrame(perf_enhanced))

### 1. Hyperparameter Tuning (Optimasi Model)
Kita akan mengoptimalkan parameter `changepoint_prior_scale` untuk menentukan seberapa fleksibel model dalam mengikuti perubahan tren.

In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics

# Contoh tuning untuk satu kategori (misal: Technology) agar cepat
# Anda bisa mengulangi ini untuk kategori lain
target_cat = 'Technology'
df_tune = df_cat[df_cat['Category'] == target_cat].copy().dropna()

param_grid = {
    'changepoint_prior_scale': [0.001, 0.05, 0.5]
}

best_params = {}
for cps in param_grid['changepoint_prior_scale']:
    m = Prophet(changepoint_prior_scale=cps)
    m.fit(df_tune[['ds', 'y']])
    # Cross validation: 365 hari training, prediksi 30 hari kedepan
    df_cv = cross_validation(m, initial='365 days', period='30 days', horizon='30 days')
    df_p = performance_metrics(df_cv)
    best_params[cps] = df_p['mae'].values.mean()

optimal_cps = min(best_params, key=best_params.get)
print(f"Nilai optimal changepoint_prior_scale untuk {target_cat} adalah: {optimal_cps}")

### 2. Simulasi Stok Inventori Berdasarkan Prediksi
Kita akan menggabungkan hasil forecast dengan data inventory untuk menghitung status stok.

In [ ]:
inventory_df = pd.read_csv('/content/inventory_summary.csv')

stock_analysis = []

for cat in categories:
    # Ambil rata-rata prediksi 7 hari kedepan
    avg_forecast_demand = results_enhanced[cat]['forecast']['yhat'].head(7).mean()
    inv_data = inventory_df[inventory_df['Category'] == cat].iloc[0]

    # Hitung status pemesanan
    reorder_point = inv_data['Reorder_Point']
    current_stock_est = inv_data['Safety_Stock'] * 1.5 # Estimasi stok saat ini

    status = "AMAN" if current_stock_est > reorder_point else "PESAN SEKARANG"

    stock_analysis.append({
        'Category': cat,
        'Predicted_Daily_Demand': avg_forecast_demand,
        'Reorder_Point': reorder_point,
        'Status': status
    })

display(pd.DataFrame(stock_analysis))

### Analisis Skala Error (Relative MAE)
Kita akan membandingkan MAE dengan rata-rata nilai aktual untuk menentukan seberapa signifikan error tersebut dalam skala bisnis.

In [ ]:
scale_analysis = []

for cat, data in results_enhanced.items():
    actual = data['actual']
    mae = mean_absolute_error(actual, data['forecast']['yhat'].values)
    mean_actual = np.mean(actual)

    # Menghitung Persentase Error terhadap Rata-rata
    relative_mae = (mae / mean_actual) * 100 if mean_actual > 0 else np.nan

    scale_analysis.append({
        'Category': cat,
        'MAE': mae,
        'Mean_Actual': mean_actual,
        'Relative_MAE (%)': relative_mae
    })

scale_df = pd.DataFrame(scale_analysis)
display(scale_df)

print("\nInterpretasi:")
for _, row in scale_df.iterrows():
    status = "BAGUS (Kecil)" if row['Relative_MAE (%)'] < 20 else "CUKUP/BURUK (Besar)"
    print(f"- {row['Category']}: Error sebesar {row['Relative_MAE (%)']:.2f}% terhadap rata-rata. Status: {status}")

### Strategi Perbaikan: Agregasi Mingguan (Weekly Aggregation)
Untuk mengurangi efek *intermittent demand*, kita akan mengelompokkan data harian menjadi total mingguan. Ini biasanya membuat tren lebih mudah ditangkap oleh model.

In [ ]:
results_weekly = {}

for cat in categories:
    df_sub = df_cat[df_cat['Category'] == cat].copy()

    # Agregasi ke level Minggu (W)
    df_weekly = df_sub.set_index('ds').resample('W').agg({
        'y': 'sum',
        'is_holiday': 'max'
    }).reset_index()

    m_weekly = Prophet()

    # Split data 80/20
    train_size = int(len(df_weekly) * 0.8)
    train_df = df_weekly.iloc[:train_size]
    test_df = df_weekly.iloc[train_size:]

    m_weekly.fit(train_df)
    forecast = m_weekly.predict(test_df)

    actual = test_df['y'].values
    predicted = forecast['yhat'].values

    mae = mean_absolute_error(actual, predicted)
    mean_act = np.mean(actual)
    rel_mae = (mae / mean_act) * 100

    results_weekly[cat] = {
        'MAE': mae,
        'Relative_MAE (%)': rel_mae
    }

weekly_perf_df = pd.DataFrame(results_weekly).T
print("Perbandingan Error Setelah Agregasi Mingguan:")
display(weekly_perf_df)

### Visualisasi Tren Mingguan: Aktual vs Prediksi
Grafik di bawah ini membandingkan total penjualan mingguan aktual dengan hasil prediksi model Prophet untuk setiap kategori pada periode testing.

In [ ]:
import matplotlib.pyplot as plt

# Set ukuran figure
plt.figure(figsize=(15, 12))

for i, cat in enumerate(categories, 1):
    plt.subplot(3, 1, i)

    # Ambil data sub-category untuk plot
    df_sub = df_cat[df_cat['Category'] == cat].copy()
    df_weekly = df_sub.set_index('ds').resample('W').agg({'y': 'sum'}).reset_index()

    # Split sesuai logika sebelumnya (80/20)
    train_size = int(len(df_weekly) * 0.8)
    test_df_weekly = df_weekly.iloc[train_size:]

    # Plot Aktual
    plt.plot(test_df_weekly['ds'], test_df_weekly['y'], label='Aktual (Mingguan)', color='black', marker='o', markersize=4, linestyle='--')

    # Plot Prediksi (Kita perlu menjalankan ulang predict untuk mendapatkan df forecast lengkap jika belum tersimpan)
    # Namun karena kita sudah punya results_weekly, kita bisa asumsikan data tersedia atau jalankan predict singkat
    m_temp = Prophet()
    m_temp.fit(df_weekly.iloc[:train_size][['ds', 'y']])
    forecast_temp = m_temp.predict(test_df_weekly[['ds']])

    plt.plot(forecast_temp['ds'], forecast_temp['yhat'], label='Prediksi', color='blue', linewidth=2)
    plt.fill_between(forecast_temp['ds'], forecast_temp['yhat_lower'], forecast_temp['yhat_upper'], color='blue', alpha=0.2, label='Confidence Interval')

    plt.title(f'Tren Mingguan {cat}: Aktual vs Prediksi', fontsize=14)
    plt.ylabel('Total Sales')
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Analisis Mendalam: Mengapa Error Masih Tinggi?
Kita akan memvisualisasikan distribusi nilai aktual vs prediksi dan melihat sebaran residual (selisih) untuk memahami apakah error disebabkan oleh kesalahan tren atau hanya variansi data yang ekstrim.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Mengambil contoh kategori Technology untuk dianalisis
cat_to_analyze = 'Technology'
data_eval = results_enhanced[cat_to_analyze]
actual = data_eval['actual']
predicted = data_eval['forecast']['yhat'].values
residuals = actual - predicted

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Distribusi Aktual vs Prediksi
sns.kdeplot(actual, label='Actual', ax=axes[0], fill=True)
sns.kdeplot(predicted, label='Predicted', ax=axes[0], fill=True)
axes[0].set_title(f'Distribusi Nilai: Actual vs Predicted ({cat_to_analyze})')
axes[0].legend()

# Plot 2: Scatter Plot Residual
axes[1].scatter(predicted, residuals, alpha=0.5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Value')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residual Plot (Cek Homoskedastisitas)')

plt.tight_layout()
plt.show()

print(f"Standar Deviasi Data Aktual: {np.std(actual):.2f}")
print(f"Mean Absolute Error: {np.mean(np.abs(residuals)):.2f}")

In [ ]:
# Menghitung Coefficient of Variation (CV) untuk menunjukkan tingkat kesulitan prediksi
cv = (np.std(actual) / np.mean(actual)) * 100
print(f"Coefficient of Variation (CV): {cv:.2f}%")
print("--- Interpretasi ---")
if cv > 100:
    print("CV > 100%: Data sangat tidak stabil (High Volatility). Prediksi dengan akurasi tinggi (>90%) secara statistik hampir mustahil tanpa fitur eksternal yang sangat kuat.")
else:
    print("CV < 100%: Data relatif stabil.")

# Membandingkan MAE dengan Standar Deviasi
print(f"\nMAE: {np.mean(np.abs(residuals)):.2f}")
print(f"Standar Deviasi: {np.std(actual):.2f}")
print(f"Persentase MAE terhadap Std Dev: {(np.mean(np.abs(residuals))/np.std(actual))*100:.2f}%")

### Penanganan Outlier untuk Stabilisasi Model
Kita akan membatasi nilai penjualan yang melebihi `Upper Bound` yang didefinisikan dalam `outlier_summary.csv`. Strategi ini membantu model Prophet agar tidak terlalu reaktif terhadap lonjakan anomali.

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error

# 1. Muat ringkasan outlier
outliers_df = pd.read_csv('/content/outlier_summary.csv')

results_robust = {}
perf_robust = []

for cat in categories:
    # Ambil batas atas untuk kategori ini
    upper_bound = outliers_df.loc[outliers_df['Category'] == cat, 'Upper Bound'].values[0]

    # Filter data
    df_sub = df_cat[df_cat['Category'] == cat].copy()

    # 2. Clipping: Membatasi nilai ekstrim ke Upper Bound
    # Kita langsung ganti kolom 'y' agar tidak terjadi duplikasi nama kolom
    df_sub['y'] = df_sub['y'].clip(upper=upper_bound)

    # Persiapkan data untuk Prophet
    df_sub = df_sub.dropna(subset=['lag_1', 'rolling_mean_7'])
    train_size = int(len(df_sub) * 0.8)
    train_df = df_sub.iloc[:train_size]
    test_df = df_sub.iloc[train_size:]

    # 3. Latih model dengan data yang sudah di-clip
    m = Prophet(holidays=holiday_df if 'holiday_df' in locals() else None)
    if 'is_holiday' in train_df.columns: m.add_regressor('is_holiday')
    m.add_regressor('lag_1')
    m.add_regressor('rolling_mean_7')

    # Fit model
    m.fit(train_df[['ds', 'y', 'is_holiday', 'lag_1', 'rolling_mean_7']])

    # Prediksi
    forecast = m.predict(test_df[['ds', 'is_holiday', 'lag_1', 'rolling_mean_7']])

    # Evaluasi terhadap data asli (menggunakan data dari df_cat yang asli tanpa clip)
    actual_raw = df_cat[(df_cat['Category'] == cat) & (df_cat['ds'].isin(test_df['ds']))]['y'].values
    predicted = forecast['yhat'].values
    mae = mean_absolute_error(actual_raw, predicted)

    results_robust[cat] = {'model': m, 'forecast': forecast, 'actual': actual_raw}
    perf_robust.append({'Category': cat, 'MAE_Robust': mae})

# Bandingkan dengan performa sebelumnya
robust_df = pd.DataFrame(perf_robust)
enhanced_prev = pd.DataFrame(perf_enhanced)[['Category', 'MAE_Enhanced']]
comparison_df = enhanced_prev.merge(robust_df, on='Category')

display(comparison_df)
print("\nAnalisis: Jika MAE_Robust lebih kecil dari MAE_Enhanced, maka penanganan outlier berhasil meningkatkan akurasi.")

### Final Step: Integrasi Fitur Kategori Lanjutan
Kita akan melatih model menggunakan seluruh fitur yang tersedia di dataset `time_series_category_feature_engineered.csv` untuk mendapatkan performa terbaik.

In [ ]:
results_final = {}
perf_final = []

# Memuat ulang data engineered untuk memastikan semua kolom tersedia
df_eng = pd.read_csv('/content/time_series_category_feature_engineered.csv', parse_dates=['ds'])

for cat in categories:
    print(f"Training final model for: {cat}...")
    df_sub = df_eng[df_eng['Category'] == cat].copy().dropna()

    # Terapkan outlier clipping yang sukses sebelumnya
    upper_bound = outliers_df.loc[outliers_df['Category'] == cat, 'Upper Bound'].values[0]
    df_sub['y'] = df_sub['y'].clip(upper=upper_bound)

    m = Prophet(holidays=holiday_df if 'holiday_df' in locals() else None)

    # Filter regressor: Pastikan hanya kolom numerik yang dimasukkan
    # Kita mengecualikan 'ds', 'y', 'Category' dan kolom teks seperti 'day_name'
    potential_regressors = [
        col for col in df_sub.columns
        if col not in ['ds', 'y', 'Category', 'day_name']
        and pd.api.types.is_numeric_dtype(df_sub[col])
    ]

    for reg in potential_regressors:
        m.add_regressor(reg)

    train_size = int(len(df_sub) * 0.8)
    train_df = df_sub.iloc[:train_size]
    test_df = df_sub.iloc[train_size:]

    # Fit model dengan data yang sudah dibersihkan
    m.fit(train_df[['ds', 'y'] + potential_regressors])

    # Forecast
    forecast = m.predict(test_df[['ds'] + potential_regressors])

    # Evaluasi terhadap data asli
    actual_raw = df_cat[(df_cat['Category'] == cat) & (df_cat['ds'].isin(test_df['ds']))]['y'].values
    mae = mean_absolute_error(actual_raw, forecast['yhat'])

    results_final[cat] = {'model': m, 'forecast': forecast, 'actual': actual_raw}
    perf_final.append({'Category': cat, 'MAE_Final': mae})

# Tampilkan perbandingan akhir
final_comparison = comparison_df.merge(pd.DataFrame(perf_final), on='Category')
display(final_comparison)
print("\nHasil Akhir: Periksa apakah MAE_Final memberikan angka terendah dibanding tahap-tahap sebelumnya.")

### 🚀 Konsolidasi & Optimalisasi Akhir
Berdasarkan analisis, kita akan menyatukan praktik terbaik ke dalam satu fungsi terpusat untuk menghindari redundansi dan memastikan akurasi maksimal.

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error

def train_optimized_model(df_input, category_name, outlier_bounds):
    # 1. Filter & Preprocessing
    df_sub = df_input[df_input['Category'] == category_name].copy().sort_values('ds')

    # 2. Tangani Outlier (Gunakan Upper Bound dari summary)
    upper = outlier_bounds.loc[outlier_bounds['Category'] == category_name, 'Upper Bound'].values[0]
    df_sub['y'] = df_sub['y'].clip(upper=upper)

    # 3. Pilih Fitur Numerik saja (Hindari teks/day_name agar tidak error)
    # Kita hanya ambil fitur yang benar-benar relevan
    features = [col for col in df_sub.columns if col not in ['ds', 'y', 'Category', 'day_name']
                and pd.api.types.is_numeric_dtype(df_sub[col])]

    # 4. Split Train/Test
    train_idx = int(len(df_sub) * 0.8)
    train, test = df_sub.iloc[:train_idx], df_sub.iloc[train_idx:]

    # 5. Konfigurasi Model Tunggal (Clean)
    # Kita gunakan is_holiday sebagai regressor (sudah ada di data), tidak perlu holiday_df terpisah
    m = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
    for feature in features:
        m.add_regressor(feature)

    m.fit(train[['ds', 'y'] + features])

    # 6. Forecast
    forecast = m.predict(test[['ds'] + features])

    return m, forecast, test['y'].values

# Eksekusi untuk semua kategori
final_results = {}
for cat in categories:
    model, fcst, actuals = train_optimized_model(df_eng, cat, outliers_df)
    mae = mean_absolute_error(actuals, fcst['yhat'])
    final_results[cat] = {'MAE': mae, 'forecast': fcst}
    print(f"Category {cat} - Optimized MAE: {mae:.2f}")

### 📊 Visualisasi Hasil Akhir (Optimized Forecast vs Actuals)
Kita akan membandingkan performa model yang telah dioptimasi dengan data aktual untuk setiap kategori.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 15))

for i, cat in enumerate(categories, 1):
    plt.subplot(3, 1, i)

    # Ambil data forecast dan actual
    fcst = final_results[cat]['forecast']
    # Filter data asli dari df_eng untuk kategori ini yang sesuai dengan index test
    test_indices = fcst['ds']
    actual_df = df_eng[(df_eng['Category'] == cat) & (df_eng['ds'].isin(test_indices))]

    # Plot Actual
    plt.scatter(actual_df['ds'], actual_df['y'], color='red', label='Actual', s=10, alpha=0.6)

    # Plot Forecast
    plt.plot(fcst['ds'], fcst['yhat'], label='Optimized Forecast', color='blue', linewidth=2)

    # Plot Confidence Interval
    plt.fill_between(fcst['ds'], fcst['yhat_lower'], fcst['yhat_upper'], color='blue', alpha=0.15, label='Confidence Interval')

    plt.title(f'Optimized Forecast vs Actual: {cat} (MAE: {final_results[cat]["MAE"]:.2f})')
    plt.xlabel('Date')
    plt.ylabel('Sales')
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 🎯 Perhitungan Persentase Akurasi Model
Kita menghitung akurasi dengan rumus: `Accuracy % = (1 - (MAE / Mean Actual)) * 100`.
Ini menunjukkan seberapa akurat model dibandingkan dengan hanya memprediksi nilai rata-rata.

In [ ]:
accuracy_stats = []

for cat in categories:
    # Ambil nilai aktual asli untuk periode test
    fcst = final_results[cat]['forecast']
    actual_raw = df_eng[(df_eng['Category'] == cat) & (df_eng['ds'].isin(fcst['ds']))]['y'].values

    mae = final_results[cat]['MAE']
    mean_actual = np.mean(actual_raw)

    # Hitung Persentase Akurasi
    # Jika MAE > Mean, akurasi bisa negatif secara teoritis, kita batasi minimal 0% untuk interpretasi sederhana
    rel_mae = mae / mean_actual
    accuracy_pct = max(0, (1 - rel_mae) * 100)

    accuracy_stats.append({
        'Category': cat,
        'MAE': mae,
        'Mean Actual': mean_actual,
        'Accuracy (%)': accuracy_pct
    })

accuracy_df = pd.DataFrame(accuracy_stats)
display(accuracy_df)

print("\nRingkasan:")
for _, row in accuracy_df.iterrows():
    print(f"- {row['Category']}: Memiliki tingkat akurasi sebesar {row['Accuracy (%)']:.2f}%")

### Perbandingan Akurasi: Harian vs Mingguan
Kode di bawah ini membandingkan hasil akurasi model harian (yang sangat volatil) dengan model mingguan (yang lebih stabil).

In [ ]:
import pandas as pd

# Mengambil data akurasi harian dari accuracy_df
daily_summary = accuracy_df[['Category', 'Accuracy (%)']].copy()
daily_summary.columns = ['Category', 'Daily Accuracy (%)']

# Mengambil data akurasi mingguan dari weekly_perf_df
# Catatan: weekly_perf_df menggunakan Relative MAE, kita ubah ke Accuracy %
weekly_summary = weekly_perf_df.reset_index()
weekly_summary.columns = ['Category', 'MAE_Weekly', 'Rel_MAE_Weekly']
weekly_summary['Weekly Accuracy (%)'] = 100 - weekly_summary['Rel_MAE_Weekly']

# Gabungkan untuk perbandingan
comparison_final = daily_summary.merge(weekly_summary[['Category', 'Weekly Accuracy (%)']], on='Category')

display(comparison_final)

print("\nKesimpulan:")
print("Akurasi Mingguan jauh lebih tinggi karena agregasi membantu menghilangkan 'noise' dari transaksi harian.")